# 3.2b RAG: Knowledge Base, retrieval pipelines and agent memory

**Story:** Customers ask the SafeDrive assistant about deductibles, exclusions and the claims process.
The answers live in five policy documents. You index them in a **Bedrock Knowledge Base** on
**S3 Vectors**, tune retrieval, build a retrieval pipeline, measure it, and then give the agent
**state**: session attributes, a knowledge base and long-term memory.

**Task 3.1** Manage deployment infrastructure. **Task 3.2** Provision and configure resources.

| Skill | What | Status |
|---|---|---|
| 3.2.7 | Create and manage Bedrock knowledge bases (vector store, indexing, retrieval optimization) | ✅ |
| 3.1.8 | RAG configurations: retrieval strategies, metadata filters, reranking | ✅ |
| 3.2.8 | Retrieval pipelines: Retrieve, RetrieveAndGenerate, custom pipeline | ✅ |
| 2.3.10 | RAG monitoring: hit rate on the KB ✅, Bedrock RAG evaluation job 🔒 | ✅ / 🔒 |
| 3.2.9 | Agent state management: session attributes, AgentCore Memory | ✅ |

## Key points for the exam

- A Knowledge Base = **data source** (S3, web, Confluence ...) → **parse** → **chunk** → **embed** →
  **vector store**. New or changed documents are picked up only by an **ingestion job** (sync).
- The **embedding model and its dimension are fixed** when the KB is created. The **chunking strategy
  is fixed** when the data source is created. To change them, create a new one and re-ingest.
- `Retrieve` returns chunks only. `RetrieveAndGenerate` returns an answer **with citations**.
- Retrieval levers: `numberOfResults`, **metadata filters**, **hybrid search** (keyword + vector, only
  on stores that support it), **reranking**, query decomposition.
- Metadata comes from a sidecar file `<document>.metadata.json` next to each document.
- **Short-term memory** = the conversation of one session. **Long-term memory** = facts and
  preferences kept across sessions.
- Bedrock Agents: `sessionId` keeps the conversation, `sessionAttributes` go to your Lambda,
  `promptSessionAttributes` go into the prompt.

| Problem | Lever |
|---|---|
| Right document is not in the top results | better chunking, hybrid search, more results + reranking |
| Results from the wrong product or year | metadata filter |
| Long question with several parts | query decomposition |
| Answer not grounded in the documents | guardrail contextual grounding check, citations, lower temperature |

## Setup

In [ ]:
import sys; sys.path.append("..")
import json, os, time, uuid
from datetime import datetime, timezone
import pandas as pd
from common import utils as U, catchup as C
print(U.REGION, U.bucket())

Catch-up: raw data, the guardrail and the Bedrock agent from 3.1b (each returns at once if it exists).

In [ ]:
C.ensure_raw_data()
guardrail_id, guardrail_version = C.ensure_guardrail()
agent_id, agent_alias_id = C.ensure_agent()
print("guardrail:", guardrail_id, guardrail_version, "| agent:", agent_id, agent_alias_id)

Clients and a small "does it exist?" helper used by the reuse checks.

In [ ]:
s3v = U.client("s3vectors")
agent = U.client("bedrock-agent")
rt = U.client("bedrock-agent-runtime")
brt = U.client("bedrock-runtime")
bedrock = U.client("bedrock")
ROLE = U.role_arn()

def exists(fn, **kwargs):
    try:
        fn(**kwargs)
        return True
    except Exception:  # noqa: BLE001
        return False

## 3.2.7 Build the Knowledge Base on S3 Vectors

- Build order: **vector bucket → vector index → knowledge base → data source → ingestion job**.
- Names and state keys match `C.ensure_kb`, so each step reuses what already exists.

| Vector store | Choose it when |
|---|---|
| **S3 Vectors** | Lowest cost, large and rarely queried data, sub-second latency is fine. Semantic search only |
| **OpenSearch Serverless** | Low latency, high query rates, **hybrid search**, rich filters. Has a minimum hourly cost |
| **Aurora PostgreSQL (pgvector)** | You already run Aurora and want vectors next to relational data (SQL joins) |
| **Neptune Analytics (GraphRAG)** | Answers need relationships across documents (entities and links) |
| Pinecone, Redis Enterprise, MongoDB Atlas | You already use that product |

**📘 Concept: What a Knowledge Base does at ingestion time and at query time**

A Knowledge Base is a managed RAG pipeline with two separate phases.

**Ingestion (runs only when you start a sync):**
1. Read the documents from the data source.
2. **Parse** them into text (default parser, or an FM / Bedrock Data Automation for tables and images).
3. **Chunk** the text into pieces.
4. **Embed** each chunk with the embedding model.
5. Write the vector, the chunk text and the metadata to the **vector store**.

**Query (every `Retrieve` call):** embed the question with the **same** embedding model, find the
nearest vectors, return their chunks.

Because questions and chunks must be embedded by the same model, some choices cannot be edited later:

| Fixed when you create ... | What is fixed | To change it |
|---|---|---|
| the vector index | dimension, distance metric | new index |
| the knowledge base | embedding model, vector store | new knowledge base, ingest again |
| the data source | chunking and parsing strategy | new data source, ingest again |

What you *can* change at any time: the documents and their metadata (then sync), and every
query-time setting (number of results, filters, reranking, the generation model and prompt).
Exam cue: "retrieval quality is poor because chunks cut sentences or tables apart" → a different
chunking or parsing strategy, which means a new data source and a full re-ingestion.

Look at the source documents. Each `.md` has a `.md.metadata.json` sidecar file.

In [ ]:
keys = U.list_keys(f"{C.RAW}/policy_docs/")
print("\n".join(os.path.basename(k) for k in keys))

In [ ]:
print(U.read_bytes(f"{C.RAW}/policy_docs/02_exclusions.md.metadata.json").decode())

### Step 1: vector bucket
A vector bucket is a new S3 bucket type made for vectors. Create it if it is missing.

In [ ]:
VBUCKET = f"safedrive-vectors-{U.account_id()}"
if not exists(s3v.get_vector_bucket, vectorBucketName=VBUCKET):
    s3v.create_vector_bucket(vectorBucketName=VBUCKET)
print(s3v.get_vector_bucket(vectorBucketName=VBUCKET)["vectorBucket"]["vectorBucketArn"])

### Step 2: vector index
- `dimension` must equal the embedding size (Titan Text Embeddings V2 = 1024).
- Metadata is **filterable by default** (max 2 KB per vector). The chunk text that Bedrock stores is
  large, so it must be declared **non-filterable**.

In [ ]:
if not exists(s3v.get_index, vectorBucketName=VBUCKET, indexName=C.VECTOR_INDEX):
    s3v.create_index(
        vectorBucketName=VBUCKET, indexName=C.VECTOR_INDEX, dataType="float32",
        dimension=U.EMBED_DIM, distanceMetric="cosine",
        metadataConfiguration={"nonFilterableMetadataKeys": ["AMAZON_BEDROCK_TEXT", "AMAZON_BEDROCK_METADATA"]})
index = s3v.get_index(vectorBucketName=VBUCKET, indexName=C.VECTOR_INDEX)["index"]
index_arn = index["indexArn"]
print(index["dimension"], index["distanceMetric"], index_arn)

### Step 3: knowledge base
Reuse the KB from state, or one with the same name.

In [ ]:
kb_id = U.get_state("kb_id")
if not kb_id:
    found = [k for k in agent.list_knowledge_bases(maxResults=100)["knowledgeBaseSummaries"] if k["name"] == C.KB_NAME]
    kb_id = found[0]["knowledgeBaseId"] if found else None
print("existing knowledge base:", kb_id)

Create it: the **embedding model** and the **storage configuration** are the two key choices.

In [ ]:
if not kb_id:
    kb_id = agent.create_knowledge_base(
        name=C.KB_NAME, roleArn=ROLE, description="SafeDrive motor policy documents",
        knowledgeBaseConfiguration={"type": "VECTOR", "vectorKnowledgeBaseConfiguration": {
            "embeddingModelArn": U.model_arn(U.EMBED_MODEL)}},
        storageConfiguration={"type": "S3_VECTORS", "s3VectorsConfiguration": {"indexArn": index_arn}},
        tags={t["Key"]: t["Value"] for t in U.TAGS})["knowledgeBase"]["knowledgeBaseId"]
print("knowledge base id:", kb_id)

In [ ]:
U.wait_until(lambda: agent.get_knowledge_base(knowledgeBaseId=kb_id)["knowledgeBase"]["status"],
             ok=("ACTIVE",), bad=("FAILED",), delay=5, label="knowledge base")

### Step 4: data source with a chunking configuration
The data source points to the S3 prefix and says how to split documents. Here: fixed-size chunks of
300 tokens with 20% overlap. (If the catch-up built the data source it uses the default chunking,
which is also fixed-size 300 tokens.)

In [ ]:
sources = agent.list_data_sources(knowledgeBaseId=kb_id)["dataSourceSummaries"]
ds = [d for d in sources if d["name"] == "policy-docs"]
ds_id = ds[0]["dataSourceId"] if ds else None
print("existing data source:", ds_id)

In [ ]:
if not ds_id:
    ds_id = agent.create_data_source(
        knowledgeBaseId=kb_id, name="policy-docs",
        dataSourceConfiguration={"type": "S3", "s3Configuration": {
            "bucketArn": f"arn:aws:s3:::{U.bucket()}", "inclusionPrefixes": [f"{C.RAW}/policy_docs/"]}},
        vectorIngestionConfiguration={"chunkingConfiguration": {
            "chunkingStrategy": "FIXED_SIZE",
            "fixedSizeChunkingConfiguration": {"maxTokens": 300, "overlapPercentage": 20}}},
    )["dataSource"]["dataSourceId"]
U.save_state(kb_id=kb_id, kb_data_source_id=ds_id, vector_bucket=VBUCKET, vector_index=C.VECTOR_INDEX)
print("data source id:", ds_id)

Read back the chunking configuration that is in force.

In [ ]:
cfg = agent.get_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)["dataSource"]
print(json.dumps(cfg.get("vectorIngestionConfiguration", "default chunking"), indent=2))

### Step 5: ingestion job (sync)
Start a sync. It is **incremental**: only new, changed or deleted documents are processed.

In [ ]:
job_id = agent.start_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id)["ingestionJob"]["ingestionJobId"]
print("ingestion job:", job_id)

In [ ]:
get_job = lambda: agent.get_ingestion_job(  # noqa: E731
    knowledgeBaseId=kb_id, dataSourceId=ds_id, ingestionJobId=job_id)["ingestionJob"]
U.wait_until(lambda: get_job()["status"], ok=("COMPLETE",), bad=("FAILED", "STOPPED"), delay=10, label="ingestion job")

Look at the statistics: documents scanned, newly indexed, failed. On a second run "new" is 0.

In [ ]:
print(json.dumps(get_job()["statistics"], indent=2))
print("failures:", get_job().get("failureReasons"))

> 🖥️ **Console check:** *Amazon Bedrock → Knowledge Bases → `safedrive-policy-kb`*.
> Look at the embedding model and vector store, open the data source `policy-docs` to see its chunking
> strategy and sync history, and try a question in the test panel. The vectors themselves are under
> *Amazon S3 → Vector buckets*.

Peek into the vector index: each vector is one chunk. The metadata holds the chunk text, the source
URI and your own attributes (`category`, `product`, `year`).

In [ ]:
vectors = s3v.list_vectors(vectorBucketName=VBUCKET, indexName=C.VECTOR_INDEX, maxResults=3, returnMetadata=True)["vectors"]
for v in vectors:
    print(v["key"], "->", sorted(v["metadata"].keys()))

### Chunking and parsing options (notes)

| Option | What it does | Use when |
|---|---|---|
| `FIXED_SIZE` | N tokens with overlap | Default. Simple, uniform text |
| `HIERARCHICAL` | Small child chunks for search, larger parent chunk returned | Answers need more surrounding context |
| `SEMANTIC` | Splits where the meaning changes (uses an FM, extra cost) | Mixed topics inside one document |
| `NONE` | One chunk per file | Documents are already split |
| Custom transformation Lambda | Your own chunking code | Special formats |
| Parsing: **foundation model** / **Bedrock Data Automation** | Reads tables, figures, scanned PDFs, images | Complex or multimodal documents |

```python
vectorIngestionConfiguration={
  "chunkingConfiguration": {"chunkingStrategy": "HIERARCHICAL", "hierarchicalChunkingConfiguration": {
      "levelConfigurations": [{"maxTokens": 1500}, {"maxTokens": 300}], "overlapTokens": 60}},
  "parsingConfiguration": {"parsingStrategy": "BEDROCK_DATA_AUTOMATION",
      "bedrockDataAutomationConfiguration": {"parsingModality": "MULTIMODAL"}}}
```

## 3.1.8 RAG configurations: retrieval strategies and reranking

- `numberOfResults`: more results = better recall, more tokens and more noise.
- **Metadata filter**: restricts the search to matching chunks before ranking.
- **Search type**: `SEMANTIC` (vectors) or `HYBRID` (vectors + keywords). Hybrid helps with exact
  terms such as IDs and product codes, but the vector store must support it.
- **Reranking**: a second model re-scores the retrieved chunks against the question. Retrieve many,
  keep the best few.

**📘 Concept: Semantic vs hybrid search, and two-stage retrieval with a reranker**

**Semantic search** compares the *meaning* of the question with the meaning of each chunk (vector
distance). It finds "car was stolen" for a question about "vehicle theft", but it is weak on exact
tokens such as a claim ID, a part number or a rare product name, because those carry little meaning.
**Keyword search** is the opposite: exact words, no understanding. **Hybrid search** runs both and
merges the two result lists, so it needs a vector store that also has a text index.

**Reranking** fixes a different weakness. Vector search is fast because question and chunk are
embedded *separately*, which makes the ranking approximate. A reranker model reads the question and
one chunk *together* and gives a much better relevance score, but it is too slow to run over the whole
index. So retrieval is done in two stages:

1. **Recall stage**: vector (or hybrid) search fetches a generous candidate set, for example 20 chunks.
2. **Precision stage**: the reranker re-orders the candidates and only the best few go into the prompt.

| Symptom in the question | Think |
|---|---|
| Exact codes, IDs or names are missed | hybrid search |
| Right chunk is retrieved but ranked low, or the prompt is full of irrelevant chunks | reranking |
| Results come from the wrong product, tenant or year | metadata filter |
| The right chunk is never retrieved at all | more results, better chunking or embeddings (a reranker cannot help) |

A helper that calls `Retrieve` and one that shows the results as a table.

In [ ]:
def kb_retrieve(query, k=3, **vector_config):
    cfg = {"vectorSearchConfiguration": {"numberOfResults": k, **vector_config}}
    return rt.retrieve(knowledgeBaseId=kb_id, retrievalQuery={"text": query},
                       retrievalConfiguration=cfg)["retrievalResults"]

def show(results):
    return pd.DataFrame([{"score": round(r.get("score", 0), 3),
                          "source": os.path.basename(r["location"]["s3Location"]["uri"]),
                          "category": r["metadata"].get("category"),
                          "text": r["content"]["text"][:90].replace("\n", " ")} for r in results])

Basic semantic retrieval. Check that the top result comes from the claims-process document.

In [ ]:
show(kb_retrieve("What is the standard deductible per claim?", k=3))

`numberOfResults`: ask for more and watch the scores fall. Low-score chunks are mostly noise.

In [ ]:
show(kb_retrieve("What is the standard deductible per claim?", k=8))

**Metadata filter** on `category`. Only chunks from the exclusions document can come back.

In [ ]:
flt = {"equals": {"key": "category", "value": "exclusions"}}
show(kb_retrieve("What happens if I report a claim late?", k=3, filter=flt))

Filters can be combined with `andAll` / `orAll` and compared with `greaterThan`, `in`, `startsWith` ...

In [ ]:
flt = {"andAll": [{"equals": {"key": "product", "value": "motor"}},
                  {"greaterThanOrEquals": {"key": "year", "value": 2026}}]}
show(kb_retrieve("Are glass claims limited?", k=3, filter=flt))

**Search type.** Force `HYBRID`. S3 Vectors supports semantic search only, so expect an error here
(if you get results instead, the service fell back to semantic search). Classic exam point:
hybrid search needs a store with text search, such as OpenSearch Serverless.

In [ ]:
try:
    display(show(kb_retrieve("claim C0000001 deductible", k=3, overrideSearchType="HYBRID")))
except Exception as e:  # noqa: BLE001
    print("HYBRID not supported by this vector store:\n", str(e)[:300])

### Reranking
Option 1: the standalone **`rerank`** API. Retrieve 8 candidates, then let the reranker order them.
Reranker models are not in every Region, so the call is guarded.

In [ ]:
RERANK_ARN = U.model_arn(U.RERANK_MODEL)

def rerank(question, chunks, top_n=3):
    """Return the top_n chunks ordered by the rerank model (falls back to the vector score)."""
    try:
        resp = rt.rerank(
            queries=[{"type": "TEXT", "textQuery": {"text": question}}],
            sources=[{"type": "INLINE", "inlineDocumentSource": {
                "type": "TEXT", "textDocument": {"text": c["content"]["text"]}}} for c in chunks],
            rerankingConfiguration={"type": "BEDROCK_RERANKING_MODEL", "bedrockRerankingConfiguration": {
                "numberOfResults": min(top_n, len(chunks)), "modelConfiguration": {"modelArn": RERANK_ARN}}})
        return [dict(chunks[r["index"]], score=r["relevanceScore"]) for r in resp["results"]]
    except Exception as e:  # noqa: BLE001
        print("reranker not available here, using vector score:", str(e)[:120])
        return sorted(chunks, key=lambda c: -c.get("score", 0))[:top_n]

Compare the order before and after reranking. The score column now holds the relevance score.

In [ ]:
question = "Can my claim be rejected if I tell SafeDrive about the accident five weeks later?"
candidates = kb_retrieve(question, k=8)
display(show(candidates[:3]))
display(show(rerank(question, candidates, top_n=3)))

Option 2: let the Knowledge Base rerank inside `Retrieve` with `rerankingConfiguration`.

In [ ]:
rerank_cfg = {"type": "BEDROCK_RERANKING_MODEL", "bedrockRerankingConfiguration": {
    "modelConfiguration": {"modelArn": RERANK_ARN}, "numberOfRerankedResults": 3}}
try:
    display(show(kb_retrieve(question, k=8, rerankingConfiguration=rerank_cfg)))
except Exception as e:  # noqa: BLE001
    print("built-in reranking not available here:", str(e)[:200])

## 3.2.8 Retrieval pipelines

| Pipeline | You get | Use when |
|---|---|---|
| `Retrieve` | chunks | You write the prompt and call the model yourself |
| `RetrieveAndGenerate` | answer + citations, session, guardrail | Standard Q&A with the least code |
| Custom pipeline | full control (rewrite, multi-query, rerank, own prompt) | Special business rules |
| Agent + knowledge base | the agent decides when to search | Search is one of several tools |

### RetrieveAndGenerate with citations
One call: retrieve, build the prompt, generate. Look at `output.text` first.

In [ ]:
MODEL_ARN = U.model_arn(U.TEXT_MODEL)
kb_cfg = {"knowledgeBaseId": kb_id, "modelArn": MODEL_ARN,
          "retrievalConfiguration": {"vectorSearchConfiguration": {"numberOfResults": 4}}}
resp = rt.retrieve_and_generate(
    input={"text": "How much deductible does a 22 year old driver pay per claim?"},
    retrieveAndGenerateConfiguration={"type": "KNOWLEDGE_BASE", "knowledgeBaseConfiguration": kb_cfg})
print(resp["output"]["text"])

**Citations** link each part of the answer to the chunks that support it. Show them to users and use
them to check grounding.

In [ ]:
for c in resp["citations"]:
    part = c["generatedResponsePart"]["textResponsePart"]["text"]
    refs = [os.path.basename(r["location"]["s3Location"]["uri"]) for r in c["retrievedReferences"]]
    print(f"- {part[:90]!r}\n    sources: {refs}")

Add a **guardrail** and your own **prompt template**. The guardrail blocks the denied topic
"legal advice", so `guardrailAction` should be `INTERVENED`.

In [ ]:
template = ("Answer the question using only these search results. If they do not contain the answer, "
            "say you do not know.\n$search_results$\nAnswer in at most two sentences.")
gen_cfg = {"promptTemplate": {"textPromptTemplate": template},
           "guardrailConfiguration": {"guardrailId": guardrail_id, "guardrailVersion": guardrail_version},
           "inferenceConfig": {"textInferenceConfig": {"temperature": 0.0, "maxTokens": 300}}}
resp = rt.retrieve_and_generate(
    input={"text": "Should I sue the other driver after my accident?"},
    retrieveAndGenerateConfiguration={"type": "KNOWLEDGE_BASE", "knowledgeBaseConfiguration": {
        **kb_cfg, "generationConfiguration": gen_cfg}})
print(resp.get("guardrailAction"), "|", resp["output"]["text"])

Built-in **query decomposition**: Bedrock splits a multi-part question into sub-queries before
retrieving. `sessionId` from the response can be passed back in to continue the conversation.

In [ ]:
resp = rt.retrieve_and_generate(
    input={"text": "What is the deductible for a young driver and how many glass claims can I make per year?"},
    retrieveAndGenerateConfiguration={"type": "KNOWLEDGE_BASE", "knowledgeBaseConfiguration": {
        **kb_cfg, "orchestrationConfiguration": {"queryTransformationConfiguration": {"type": "QUERY_DECOMPOSITION"}}}})
print(resp["output"]["text"])
print("session:", resp["sessionId"])

### A custom pipeline: rewrite → retrieve → rerank → generate
Step 1, **query rewrite / decomposition** with a small, cheap model.

In [ ]:
def rewrite(question):
    """Split a question into 1-3 standalone search queries (falls back to the question itself)."""
    prompt = ("Split the user question into 1 to 3 short standalone search queries about a motor "
              "insurance policy. Return only a JSON list of strings.\nQuestion: " + question)
    text = U.converse(prompt, model_id=U.SMALL_MODEL, temperature=0)
    try:
        return json.loads(text[text.index("["): text.rindex("]") + 1])
    except ValueError:
        return [question]

question = "I am 23 and my windscreen cracked. What deductible applies and is there a limit on glass claims?"
queries = rewrite(question)
print(queries)

Step 2, **retrieve** for every sub-query and remove duplicate chunks.

In [ ]:
def retrieve_many(queries, k=4):
    unique = {}
    for q in queries:
        for r in kb_retrieve(q, k=k):
            unique.setdefault(r["content"]["text"], r)
    return list(unique.values())

candidates = retrieve_many(queries)
print(len(candidates), "unique chunks from", len(queries), "queries")

Step 3, **rerank** against the original question and keep the best four.

In [ ]:
top = rerank(question, candidates, top_n=4)
show(top)

Step 4, **generate** with the Converse API. The prompt numbers the chunks so the model can cite them.

In [ ]:
def generate(question, chunks):
    context = "\n\n".join(f"[{i + 1}] ({os.path.basename(c['location']['s3Location']['uri'])}) {c['content']['text']}"
                          for i, c in enumerate(chunks))
    resp = brt.converse(
        modelId=U.TEXT_MODEL,
        system=[{"text": "Answer only from the context. Cite sources like [1]. If unsure, say you do not know."}],
        messages=[{"role": "user", "content": [{"text": f"Context:\n{context}\n\nQuestion: {question}"}]}],
        inferenceConfig={"maxTokens": 300, "temperature": 0.0})
    return resp["output"]["message"]["content"][0]["text"], resp["usage"]

answer, usage = generate(question, top)
print(answer, "\n", usage)

The whole pipeline as one function, ready to put behind an API or a Lambda.

In [ ]:
def rag_pipeline(question, k=4, top_n=4):
    chunks = rerank(question, retrieve_many(rewrite(question), k=k), top_n=top_n)
    return generate(question, chunks)[0]

print(rag_pipeline("Is damage covered if I use my private car for food delivery?"))

## 2.3.10 RAG evaluation on the Knowledge Base

- Evaluate **retrieval** (did the right chunk come back?) and **generation** (is the answer correct
  and grounded?) separately.
- Retrieval metrics you compute yourself: **hit rate@k** and **MRR** against a labelled question set.
- **Bedrock RAG evaluation jobs** use an LLM judge: context relevance and coverage (retrieve only),
  correctness, completeness, faithfulness, citation precision (retrieve and generate).
- Re-run the evaluation after every change to chunking, the embedding model or the documents.

Load the golden questions. Each has the expected answer and the source document.

In [ ]:
qa = U.read_jsonl(f"{C.RAW}/qa_eval.jsonl")
print(len(qa), "questions")
print(qa[0])

Retrieve the top 5 for every question and record the rank of the first chunk from the right document.

In [ ]:
def first_hit_rank(item, k=5):
    sources = [os.path.basename(r["location"]["s3Location"]["uri"]) for r in kb_retrieve(item["question"], k=k)]
    return sources.index(item["source"]) + 1 if item["source"] in sources else None

ranks = [first_hit_rank(item) for item in qa]
print(ranks)

Hit rate@k = share of questions with the right document in the top k. MRR = mean of 1/rank.

In [ ]:
metrics = {f"hit_rate@{k}": sum(1 for r in ranks if r and r <= k) / len(ranks) for k in (1, 3, 5)}
metrics["mrr"] = sum(1 / r for r in ranks if r) / len(ranks)
pd.Series(metrics).round(3)

Write the dataset in the format a Bedrock RAG evaluation job expects (prompt + reference answer).

In [ ]:
lines = [json.dumps({"conversationTurns": [{
    "prompt": {"content": [{"text": item["question"]}]},
    "referenceResponses": [{"content": [{"text": item["answer"]}]}]}]}) for item in qa]
eval_key = f"{U.PREFIX}/genai/rag_eval/dataset.jsonl"
U.client("s3").put_object(Bucket=U.bucket(), Key=eval_key, Body="\n".join(lines))
print(U.s3_uri(eval_key), "\n", lines[0][:160])

🔒 **Bedrock RAG evaluation job.** 💰 Runs the KB and a judge model on every question (a few cents here,
10-20 minutes). The evaluator must be a model that Bedrock supports as a judge in your Region, and the
S3 bucket needs CORS enabled if you want to view the report in the console.

In [ ]:
if U.expensive("Bedrock RAG evaluation job"):
    eval_job_arn = bedrock.create_evaluation_job(
        jobName=f"safedrive-rag-eval-{int(time.time())}", roleArn=ROLE, applicationType="RagEvaluation",
        inferenceConfig={"ragConfigs": [{"knowledgeBaseConfig": {"retrieveAndGenerateConfig": {
            "type": "KNOWLEDGE_BASE", "knowledgeBaseConfiguration": kb_cfg}}}]},
        evaluationConfig={"automated": {
            "datasetMetricConfigs": [{
                "taskType": "General", "dataset": {"name": "safedrive-qa", "datasetLocation": {"s3Uri": U.s3_uri(eval_key)}},
                "metricNames": ["Builtin.Correctness", "Builtin.Faithfulness", "Builtin.CitationPrecision"]}],
            "evaluatorModelConfig": {"bedrockEvaluatorModels": [{"modelIdentifier": U.LARGE_MODEL}]}}},
        outputDataConfig={"s3Uri": U.s3_uri(f"{U.PREFIX}/genai/rag_eval/output/")},
        jobTags=[{"key": t["Key"], "value": t["Value"]} for t in U.TAGS])["jobArn"]
    print(eval_job_arn)

🔒 Check the job. When it is `Completed`, the scores per metric are in the output S3 prefix and in
the console under *Evaluations*.

In [ ]:
if U.expensive("Bedrock RAG evaluation job status"):
    job = bedrock.get_evaluation_job(jobIdentifier=eval_job_arn)
    print(job["status"], job.get("failureMessages"))
    print(U.list_keys(f"{U.PREFIX}/genai/rag_eval/output/")[:5])

## 3.2.9 Agent state management

| State | Lives where | Lifetime |
|---|---|---|
| Conversation history | Bedrock Agents session (`sessionId`) | until `idleSessionTTLInSeconds` or `endSession` |
| `sessionAttributes` | passed to action-group Lambdas, **not** shown to the model | the session |
| `promptSessionAttributes` | injected into the prompt | one turn |
| Bedrock Agents memory (`SESSION_SUMMARY`, `memoryId`) | summaries of past sessions | up to 365 days |
| **AgentCore Memory** | events (short-term) and extracted records (long-term) | configurable |
| Bedrock **session management APIs** (`create_session`, `put_invocation_step`) | checkpoints for LangGraph and other frameworks | until deleted |
| DynamoDB / ElastiCache | your own state store (Strands `S3SessionManager`, custom tables) | you decide |

Helper to call the Bedrock agent (same as in 3.1b).

In [ ]:
def ask_agent(text, session_id, alias_id=None, **kwargs):
    resp = rt.invoke_agent(agentId=agent_id, agentAliasId=alias_id or agent_alias_id, sessionId=session_id,
                           inputText=text, enableTrace=True, **kwargs)
    answer, traces = "", []
    for event in resp["completion"]:
        if "chunk" in event:
            answer += event["chunk"]["bytes"].decode()
        if "trace" in event:
            traces.append(event["trace"]["trace"])
    return answer, traces

### Session continuity with `sessionId`
Turn 1 names the claim. Turn 2 says "it" and still works, because both turns share one session.

In [ ]:
session = str(uuid.uuid4())
print(ask_agent("What is the status of claim C0000005?", session)[0])
print(ask_agent("And how much was claimed on it?", session)[0])

A **new** `sessionId` starts with an empty history, so the agent has to ask which claim you mean.

In [ ]:
print(ask_agent("And how much was claimed on it?", str(uuid.uuid4()))[0])

### `sessionAttributes` and `promptSessionAttributes`
- `promptSessionAttributes`: facts the model should see this turn (customer name, today's date,
  the claim on screen).
- `sessionAttributes`: values for your tools (customer ID, auth context). The Lambda reads them from
  `event["sessionAttributes"]`. The model does not see them.

In [ ]:
state = {"sessionAttributes": {"customer_id": "CUST-0042", "channel": "app"},
         "promptSessionAttributes": {"customer_name": "Priya", "claim_on_screen": "C0000006"}}
answer, _ = ask_agent("Hello, what is the status of the claim on my screen?", str(uuid.uuid4()), sessionState=state)
print(answer)

If the answer ignores the attributes, the model's default orchestration prompt has no
`$prompt_session_attributes$` placeholder. Add it with a prompt override (`promptOverrideConfiguration`).

### Attach the Knowledge Base to the agent
Now the agent can decide between the claim-lookup tool and the policy documents. The description
tells the model when to search. This changes the **DRAFT**, so prepare again.

In [ ]:
attached = agent.list_agent_knowledge_bases(agentId=agent_id, agentVersion="DRAFT")["agentKnowledgeBaseSummaries"]
if not any(k["knowledgeBaseId"] == kb_id for k in attached):
    agent.associate_agent_knowledge_base(
        agentId=agent_id, agentVersion="DRAFT", knowledgeBaseId=kb_id, knowledgeBaseState="ENABLED",
        description="SafeDrive motor policy rules: coverage, exclusions, deductibles, claims process, discounts.")
print("knowledge base attached to DRAFT")

In [ ]:
agent.prepare_agent(agentId=agent_id)
U.wait_until(lambda: agent.get_agent(agentId=agent_id)["agent"]["agentStatus"],
             ok=("PREPARED",), bad=("FAILED",), delay=5, label="agent")

> 🖥️ **Console check:** *Amazon Bedrock → Agents → `safedrive-claims-agent`*, then open the working draft.
> The knowledge base is now listed next to the action group, with the description the model uses to
> decide when to search.

Test the DRAFT through the test alias `TSTALIASID`. The alias `live` still points to the old version
without the KB (versions and aliases are covered in 3.3b).

In [ ]:
answer, traces = ask_agent("How many glass claims are allowed per policy year?", str(uuid.uuid4()), alias_id="TSTALIASID")
print(answer)
lookups = [t["orchestrationTrace"]["invocationInput"]["knowledgeBaseLookupInput"] for t in traces
           if "knowledgeBaseLookupInput" in t.get("orchestrationTrace", {}).get("invocationInput", {})]
print("knowledge base lookups:", lookups)

### AgentCore Memory: short-term and long-term

- **Short-term**: you write **events** (conversation turns) per `actorId` + `sessionId` and read them
  back to rebuild the context. Works with any framework.
- **Long-term**: **strategies** extract records from the events in the background:
  `semanticMemoryStrategy` (facts), `userPreferenceMemoryStrategy` (preferences),
  `summaryMemoryStrategy` (session summaries), plus custom and episodic strategies.
- Records are stored under a **namespace** such as `/facts/{actorId}/` and found by semantic search.

**📘 Concept: Short-term vs long-term agent memory**

A foundation model is stateless: it only knows what is in the prompt of the current call. "Memory"
is whatever your system stores and puts back into the prompt.

| | Short-term memory | Long-term memory |
|---|---|---|
| Holds | The raw turns of **one conversation** | Facts, preferences and summaries distilled from **many** conversations |
| Scope | One session | One user (actor), across sessions |
| How it is read | Load the last turns in order | Semantic search for what is relevant now |
| Solves | "What did I just say?" | "You should already know this about me" |

In AgentCore Memory the two are linked. Your agent only writes **events** (short-term). A **memory
strategy** is a background job that reads new events and uses a model to extract long-term records:
facts (semantic), user preferences, or a running summary of the session. Extraction is asynchronous,
so a new fact is not available in long-term memory in the same instant it was said.

Exam cues: "remember the user across sessions, days or devices" → long-term memory with a strategy.
"Keep context inside one long conversation" → short-term memory or the session itself. A longer
session timeout or a bigger context window is never the answer to a cross-session requirement.

Create the memory resource with two long-term strategies (reuse if it exists).

In [ ]:
ac = U.client("bedrock-agentcore-control")
acd = U.client("bedrock-agentcore")
MEMORY_NAME = "safedrive_claims_memory"
mems = [m for m in ac.list_memories(maxResults=100)["memories"] if m["id"].startswith(MEMORY_NAME)]
print("existing memory:", [m["id"] for m in mems])

In [ ]:
if mems:
    memory_id = mems[0]["id"]
else:
    memory_id = ac.create_memory(
        name=MEMORY_NAME, description="SafeDrive assistant memory", eventExpiryDuration=30,
        memoryStrategies=[
            {"semanticMemoryStrategy": {"name": "claim_facts", "namespaces": ["/facts/{actorId}/"]}},
            {"userPreferenceMemoryStrategy": {"name": "customer_preferences",
                                              "namespaces": ["/preferences/{actorId}/"]}}],
        tags={t["Key"]: t["Value"] for t in U.TAGS})["memory"]["id"]
print("memory id:", memory_id)

Wait until the memory is `ACTIVE` (1-3 minutes).

In [ ]:
U.wait_until(lambda: ac.get_memory(memoryId=memory_id)["memory"]["status"],
             ok=("ACTIVE",), bad=("FAILED",), delay=15, label="memory")

> 🖥️ **Console check:** *Amazon Bedrock AgentCore → Memory → `safedrive_claims_memory`*.
> Look at the two long-term strategies and their namespaces, and the event expiry (how long short-term events are kept).

**Short-term:** store one conversation as an event. `actorId` is the customer, `sessionId` the chat.

In [ ]:
ACTOR, CHAT = "customer-priya", f"chat-{uuid.uuid4()}"
turns = [("USER", "Hi, I am Priya. My policy is P004507 and my claim is C0000001."),
         ("ASSISTANT", "Hello Priya. Claim C0000001 is on file. How can I help?"),
         ("USER", "Please always contact me by email, never by phone. I prefer short answers."),
         ("ASSISTANT", "Noted. I will use email and keep answers short.")]
event = acd.create_event(
    memoryId=memory_id, actorId=ACTOR, sessionId=CHAT, eventTimestamp=datetime.now(timezone.utc),
    payload=[{"conversational": {"role": role, "content": {"text": text}}} for role, text in turns])["event"]
print("event id:", event["eventId"])

Read the short-term memory back. An agent loads these turns at the start of each request.

In [ ]:
events = acd.list_events(memoryId=memory_id, actorId=ACTOR, sessionId=CHAT, includePayloads=True)["events"]
for e in events:
    for p in e["payload"]:
        print(p["conversational"]["role"], ":", p["conversational"]["content"]["text"])

**Long-term:** extraction runs asynchronously (about a minute). Poll with a semantic search until the
preference record appears. Note that this is a *different session*: only `actorId` links it.

In [ ]:
def recall(query, namespace, top_k=3):
    return acd.retrieve_memory_records(memoryId=memory_id, namespace=namespace,
                                       searchCriteria={"searchQuery": query, "topK": top_k})["memoryRecordSummaries"]

for _ in range(16):
    prefs = recall("How does the customer want to be contacted?", f"/preferences/{ACTOR}/")
    if prefs:
        break
    time.sleep(15)
print([p["content"]["text"] for p in prefs])

The semantic strategy extracted facts from the same conversation.

In [ ]:
facts = recall("Which policy and claim belong to this customer?", f"/facts/{ACTOR}/")
for f in facts:
    print(round(f.get("score", 0), 3), f["content"]["text"])

Use the memories: put them into the system prompt of a new conversation.

In [ ]:
memory_text = "\n".join(r["content"]["text"] for r in prefs + facts)
print(U.converse("Which claim do I have with you, and how will you contact me?",
                 system="You are the SafeDrive assistant. Known about this customer:\n" + memory_text))

📝 **Other options.** Strands has session managers (`FileSessionManager`, `S3SessionManager`) and an
AgentCore Memory session manager. Bedrock Agents can keep cross-session summaries with
`memoryConfiguration={"enabledMemoryTypes": ["SESSION_SUMMARY"], "storageDays": 30}` plus a `memoryId`
per user in `invoke_agent`. A DynamoDB table keyed by `session_id` is the do-it-yourself choice.

## Exam traps

- New documents in S3 are **not** searchable until an **ingestion job** runs. The sync is not automatic.
- You cannot change the **embedding model** of a KB or the **chunking strategy** of a data source in
  place. Create a new KB / data source and re-ingest.
- **Hybrid search** needs a vector store with text search (OpenSearch Serverless, Aurora, MongoDB
  Atlas). S3 Vectors is semantic only, and its trade-off is lowest cost for slower queries.
- Reranking improves precision but adds latency and cost. It does not help if the right chunk was
  never retrieved: raise `numberOfResults` first.
- `sessionAttributes` are for tools, `promptSessionAttributes` are for the model. Long-term memory
  needs a memory feature (AgentCore Memory, agent memory), not a longer session.

## Quiz

**1.** New policy PDFs are uploaded to the KB's S3 prefix every night, but the assistant never finds
them. What is missing?

A. A larger `numberOfResults`  B. Running `StartIngestionJob` after the upload  C. A reranker model  D. A new embedding model

<details><summary>Answer</summary>B. The KB only indexes documents during an ingestion job (sync).</details>

**2.** Users search for exact part numbers such as `WS-4471`, and semantic search returns similar but
wrong parts. Which change helps most?

A. Lower the temperature  B. Hybrid search on a vector store that supports it  C. Smaller chunks  D. Larger embedding dimension

<details><summary>Answer</summary>B. Hybrid search adds keyword matching, which is strong for exact tokens.</details>

**3.** The KB holds documents for motor, home and travel products. Motor questions sometimes get
answers from travel documents. What is the simplest fix?

A. One KB per customer  B. A metadata filter on `product`  C. Hierarchical chunking  D. Provisioned Throughput

<details><summary>Answer</summary>B. Add metadata files and filter at query time.</details>

**4.** A returning customer should not have to repeat their contact preference in every new chat.
The agent is a LangGraph agent on AgentCore Runtime. What should you add?

A. A longer `idleRuntimeSessionTimeout`  B. AgentCore Memory with a user-preference strategy  C. `promptSessionAttributes`  D. A bigger context window

<details><summary>Answer</summary>B. Long-term memory strategies extract and keep preferences across sessions.</details>

**5.** A Lambda action group needs the caller's customer ID on every tool call, but the model must not
see or change it. Where do you put it?

A. In the instruction  B. In `promptSessionAttributes`  C. In `sessionAttributes`  D. In the knowledge base

<details><summary>Answer</summary>C. Session attributes are passed to the Lambda and stay out of the prompt.</details>

## Save state

In [ ]:
state = U.save_state(kb_id=kb_id, kb_data_source_id=ds_id, vector_bucket=VBUCKET, vector_index=C.VECTOR_INDEX)
print({k: state[k] for k in ("kb_id", "kb_data_source_id", "vector_bucket", "vector_index")})

## Cleanup

**Kept for later notebooks:** the Knowledge Base `safedrive-policy-kb`, its data source, the vector
bucket and index (S3 Vectors has no idle compute cost), and the KB association on the agent's DRAFT.
The teardown notebook removes them.

Deleted here: the AgentCore Memory.

### Before you delete: what later notebooks use

The cleanup cells below **do not delete** the items in this table.

| Kept | Used by later notebooks | If you delete it anyway |
|---|---|---|
| Knowledge Base `safedrive-policy-kb`, its data source and the vector index `policy-docs` | 3.3b, 4.1b, 4.3b (and an example policy in 4.3a) | Rebuilt by the catch-up cell (a few minutes) |
| Agent and Lambda (from 3.1b) | 3.3b, 4.1b, 4.2 | Rebuilt by the catch-up cell |

- **Continuing to the next notebook soon?** Run the cleanup cells and keep the items above. None of them bills while idle (only a little storage).
- **Stopping for good?** Run `99_cleanup/99_teardown.ipynb`; it removes everything, including these.

In [ ]:
U.safe(ac.delete_memory, memoryId=memory_id)

The evaluation dataset on S3 is tiny. Remove it if you like.

In [ ]:
U.safe(U.client("s3").delete_object, Bucket=U.bucket(), Key=eval_key)